# Тестирование сохранения данных опционов в БД

Этот notebook используется для проверки работы периодического сохранения данных из WebSocket в SQLite базу данных.

## Инструкция по тестированию:

1. **Запустите приложение в Docker:**
   ```bash
   docker compose up --build
   ```

2. **Дождитесь подключения WebSocket и начала сбора данных:**
   - В логах должно появиться сообщение "✅ WebSocket connected"
   - Должно появиться "Периодическое сохранение данных в БД запущено"
   - Подождите минимум 5-10 минут, чтобы накопились данные

3. **Если БД находится в Docker контейнере, скопируйте её локально:**
   ```bash
   docker cp option-telegram-bot:/app/data/options.db ./data/options.db
   ```
   Или если БД монтируется как volume, она уже будет доступна локально.

4. **Запустите ячейки ниже для проверки данных**

## Строка подключения к БД:

**Локально:** `./data/options.db` (относительно корня проекта)  
**В Docker:** `/app/data/options.db` (внутри контейнера)

In [4]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Путь к базе данных
# В Docker контейнере база находится в /app/data/options.db
# Локально - в ./data/options.db относительно корня проекта
project_root = Path.cwd()
db_path = project_root / "data" / "options.db"

print(f"Путь к БД: {db_path}")
print(f"БД существует: {db_path.exists()}")

# Строка подключения для SQLite (просто путь к файлу)
connection_string = str(db_path)
print(f"\nСтрока подключения: {connection_string}")

Путь к БД: /home/alex/Боты/Bot_Option_cursor/data/options.db
БД существует: True

Строка подключения: /home/alex/Боты/Bot_Option_cursor/data/options.db


In [5]:
# Подключение к БД
conn = sqlite3.connect(str(db_path))
conn.row_factory = sqlite3.Row  # Для доступа к колонкам по имени

# Проверка таблиц
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Таблицы в БД:")
for table in tables:
    print(f"  - {table[0]}")

Таблицы в БД:
  - option_history
  - sqlite_sequence
  - underlying_history
  - iv_history
  - support_resistance_levels
  - agent_signals
  - signal_results


In [6]:
# 1. Проверка количества записей в option_history
query = "SELECT COUNT(*) as count FROM option_history"
df = pd.read_sql_query(query, conn)
print("Количество записей в option_history:")
print(df)

Количество записей в option_history:
   count
0      4


In [23]:
# 2. Последние 10 записей из option_history
query = """
SELECT 
    symbol,
    timestamp,
    ask_price,
    bid_price,
    mark_price,
    iv,
    delta,
    gamma,
    underlying_price
FROM option_history
ORDER BY timestamp DESC
LIMIT 10
"""
df = pd.read_sql_query(query, conn)
print("Последние 10 записей:")
print(df.to_string())

Последние 10 записей:
                     symbol            timestamp  ask_price  bid_price   mark_price      iv     delta     gamma  underlying_price
0  BTC-11JAN26-91000-P-USDT  2026-01-08T10:10:00     1785.0     1740.0  1771.186750  0.3611 -0.636383  0.000130      89936.345372
1  BTC-11JAN26-91000-C-USDT  2026-01-08T10:10:00      710.0      705.0   707.532122  0.3611  0.363617  0.000130      89936.345372
2  BTC-10JAN26-91000-P-USDT  2026-01-08T10:10:00     1720.0     1660.0  1703.952447  0.4151 -0.648512  0.000138      89924.718166
3  BTC-10JAN26-91000-C-USDT  2026-01-08T10:10:00      630.0      625.0   628.670613  0.4151  0.351488  0.000138      89924.718166
4   BTC-9JAN26-91000-P-USDT  2026-01-08T10:10:00     1440.0     1285.0  1368.337324  0.3866 -0.730177  0.000173      89914.680000
5   BTC-9JAN26-91000-C-USDT  2026-01-08T10:10:00      290.0      285.0   283.017324  0.3866  0.269823  0.000173      89914.680000
6  BTC-11JAN26-91000-P-USDT  2026-01-08T10:05:00     1750.0     1700

In [31]:
query = """
SELECT 
    *
FROM option_history
ORDER BY timestamp DESC
LIMIT 3
"""
df = pd.read_sql_query(query, conn)
print("Последние 10 записей:")
print(df.to_string())

Последние 10 записей:
   id                    symbol            timestamp  ask_price  bid_price   mark_price      iv  ask_iv  bid_iv  mark_iv     delta     gamma       vega       theta  volume_24h  open_interest  underlying_price
0  76  BTC-11JAN26-91000-P-USDT  2026-01-08T10:25:00     1620.0     1600.0  1602.290556  0.3615  0.3671  0.3606   0.3615 -0.600812  0.000133  31.026035 -193.621405        0.60           0.60      90207.934667
1  75  BTC-11JAN26-91000-C-USDT  2026-01-08T10:25:00      815.0      805.0   810.225223  0.3615  0.3629  0.3597   0.3615  0.399188  0.000133  31.026035 -193.621405        0.10           0.10      90207.934667
2  74  BTC-10JAN26-91000-P-USDT  2026-01-08T10:25:00     1545.0     1505.0  1523.804634  0.4126  0.4210  0.4050   0.4126 -0.611568  0.000143  24.912282 -271.060284       17.68          10.01      90196.522632


In [24]:
# 3. Уникальные символы в БД
query = """
SELECT DISTINCT symbol, COUNT(*) as count
FROM option_history
GROUP BY symbol
ORDER BY count DESC
"""
df = pd.read_sql_query(query, conn)
print("Уникальные символы и количество записей:")
print(df.to_string())

Уникальные символы и количество записей:
                     symbol  count
0   BTC-9JAN26-91000-C-USDT     12
1   BTC-9JAN26-91000-P-USDT     12
2  BTC-10JAN26-91000-C-USDT      9
3  BTC-10JAN26-91000-P-USDT      9
4  BTC-11JAN26-91000-C-USDT      8
5  BTC-11JAN26-91000-P-USDT      8


In [26]:
# 4. Проверка временных интервалов сохранения (должны быть кратны 5 минутам)
query = """
SELECT 
    timestamp,
    strftime('%M', timestamp) as minute,
    COUNT(*) as count
FROM option_history
GROUP BY timestamp
ORDER BY timestamp DESC
LIMIT 20
"""
df = pd.read_sql_query(query, conn)
print("Временные метки сохранения (последние 20):")
print(df.to_string())

# Проверяем, что минуты кратны 5
if len(df) > 0:
    minutes = df['minute'].astype(int)
    not_aligned = minutes[minutes % 5 != 0]
    if len(not_aligned) > 0:
        print(f"\n⚠️ ВНИМАНИЕ: Найдены записи не выровненные по 5 минутам: {not_aligned.tolist()}")
    else:
        print("\n✅ Все временные метки выровнены по 5-минутным интервалам")

Временные метки сохранения (последние 20):
              timestamp minute  count
0   2026-01-08T10:10:00     10      6
1   2026-01-08T10:05:00     05      6
2   2026-01-08T10:00:00     00      6
3   2026-01-08T09:55:00     55      6
4   2026-01-08T09:50:00     50      6
5   2026-01-08T09:45:00     45      6
6   2026-01-08T09:40:00     40      6
7   2026-01-08T09:35:00     35      6
8   2026-01-08T09:30:00     30      4
9   2026-01-08T09:25:00     25      2
10  2026-01-08T09:20:00     20      2
11  2026-01-08T09:15:00     15      2

✅ Все временные метки выровнены по 5-минутным интервалам


In [29]:
# 5. История для конкретного символа (если есть данные)
query = """
SELECT symbol
FROM option_history
LIMIT 1
"""
df_symbol = pd.read_sql_query(query, conn)

if len(df_symbol) > 0:
    test_symbol = df_symbol['symbol'].iloc[0]
    print(f"Анализ истории для символа: {test_symbol}\n")
    
    query_history = """
    SELECT 
        timestamp,
        ask_price,
        bid_price,
        mark_price,
        iv,
        delta,
        underlying_price
    FROM option_history
    WHERE symbol = ?
    ORDER BY timestamp ASC
    """
    df_history = pd.read_sql_query(query_history, conn, params=(test_symbol,))
    print(f"Количество записей: {len(df_history)}")
    print("\nПервые 5 записей:")
    print(df_history.head().to_string())
    print("\nПоследние 5 записей:")
    print(df_history.tail().to_string())
else:
    print("В БД пока нет данных")

Анализ истории для символа: BTC-10JAN26-91000-C-USDT

Количество записей: 10

Первые 5 записей:
             timestamp  ask_price  bid_price  mark_price      iv     delta  underlying_price
0  2026-01-08T09:30:00      705.0      700.0  704.395180  0.4192  0.375990      90084.800095
1  2026-01-08T09:35:00      650.0      645.0  649.143179  0.4105  0.360584      89995.479643
2  2026-01-08T09:40:00      665.0      655.0  660.373913  0.4069  0.367343      90053.581845
3  2026-01-08T09:45:00      690.0      685.0  688.335424  0.4096  0.376551      90113.124196
4  2026-01-08T09:50:00      670.0      665.0  673.509772  0.4112  0.370167      90064.956839

Последние 5 записей:
             timestamp  ask_price  bid_price  mark_price      iv     delta  underlying_price
5  2026-01-08T09:55:00      645.0      640.0  644.332148  0.4092  0.360411      90001.432569
6  2026-01-08T10:00:00      615.0      610.0  606.187729  0.4144  0.342594      89859.658804
7  2026-01-08T10:05:00      655.0      645.0 

In [28]:
# 6. Статистика по underlying_history
query = """
SELECT 
    symbol,
    COUNT(*) as count,
    MIN(timestamp) as first_record,
    MAX(timestamp) as last_record
FROM underlying_history
GROUP BY symbol
"""
df = pd.read_sql_query(query, conn)
print("Статистика по базовым активам:")
print(df.to_string())

Статистика по базовым активам:
  symbol  count         first_record          last_record
0    BTC     13  2026-01-08T09:15:00  2026-01-08T10:15:00


In [30]:
# 7. Проверка IV истории
query = """
SELECT 
    symbol,
    COUNT(*) as count,
    AVG(iv) as avg_iv,
    MIN(iv) as min_iv,
    MAX(iv) as max_iv
FROM iv_history
WHERE iv IS NOT NULL
GROUP BY symbol
ORDER BY count DESC
LIMIT 10
"""
df = pd.read_sql_query(query, conn)
print("Статистика IV по символам:")
print(df.to_string())

Статистика IV по символам:
                     symbol  count    avg_iv  min_iv  max_iv
0   BTC-9JAN26-91000-C-USDT     13  0.384908  0.3785  0.3897
1   BTC-9JAN26-91000-P-USDT     13  0.384908  0.3785  0.3897
2  BTC-10JAN26-91000-C-USDT     10  0.412190  0.4069  0.4192
3  BTC-10JAN26-91000-P-USDT     10  0.412190  0.4069  0.4192
4  BTC-11JAN26-91000-C-USDT      9  0.360356  0.3584  0.3635
5  BTC-11JAN26-91000-P-USDT      9  0.360356  0.3584  0.3635


In [ ]:
# Закрываем соединение
conn.close()
print("Соединение с БД закрыто")